# Charles Schwab TraderAPI Setup Example

schwab_data.py official reference
==============
Market-data loader for a futures trading algorithm, built on the Charles Schwab
Trader API (Market Data Production) via the `schwab-py` wrapper.
 
It exposes the data your algo needs in `datetime, open, high, low, close` (OHLC)
form, and is written for a scheduled "pipeline" run (e.g. 08:00–13:00 ET).
 
────────────────────────────────────────────────────────────────────────────
THREE DATA PATHS (pick per use case)
────────────────────────────────────────────────────────────────────────────
1. LIVE SNAPSHOT  -> get_live_quote_ohlc()
   One REST call (Client.get_quotes). Returns the *current session* OHLC plus
   last/bid/ask for the contract. Best for "fire a test order based on
   near-current market conditions." Stateless: no connection to hold open.
 
2. LIVE OHLC BARS -> stream_ohlc_bars()  [the open/receive/close model]
   Opens the Schwab streaming WebSocket, subscribes to CHART_FUTURES, and emits
   completed 1-minute OHLCV bars as they print. login() = open, logout() = close.
   This is the native datetime+OHLC feed for FUTURES.
 
3. HISTORICAL OHLC -> get_historical_ohlc()  [documented separately, see caveat]
   Uses Client.get_price_history_*. IMPORTANT: Schwab serves price history for
   EQUITIES/ETFs ONLY — NOT futures. So this path will NOT return MYM history.
   For historical futures bars you need a different vendor (Databento, CQG,
   Rithmic/Barchart, etc.). Function is included + guarded so the equities path
   is documented and ready, and the futures limitation fails loudly.
 
────────────────────────────────────────────────────────────────────────────
SYMBOLS
────────────────────────────────────────────────────────────────────────────
Schwab futures format is slash-prefixed with a 2-digit year:  /MYMU26
TradersPost (your execution leg) uses a 4-digit year:          MYMU2026
Both refer to the Micro E-mini Dow, September 2026 contract.
 
────────────────────────────────────────────────────────────────────────────
SETUP
────────────────────────────────────────────────────────────────────────────
    pip install schwab-py
Set these environment variables (do NOT hardcode secrets):
    SCHWAB_API_KEY      = your app key (consumer key)
    SCHWAB_APP_SECRET   = your app secret
    SCHWAB_CALLBACK_URL = e.g. https://127.0.0.1:8182  (must match the portal)
    SCHWAB_TOKEN_PATH   = e.g. ./schwab_token.json     (where tokens persist)
 
First run opens a browser for the one-time OAuth login. After that the token
file auto-refreshes: access token ~30 min, refresh token ~7 days. For a headless
pipeline, run the login once locally and copy the token file to the host.
"""

In [ ]:
""" 
Required Packages for running Schwab API
"""
from __future__ import annotations
 
import asyncio
import os
import logging
from datetime import datetime, time as dtime
from zoneinfo import ZoneInfo
 
from schwab import auth
from schwab.streaming import StreamClient

In [ ]:
# ── Symbol helpers ──────────────────────────────────────────────────────────
_MONTHS = "FGHJKMNQUVXZ"  # CME month codes Jan..Dec
 
def schwab_to_traderspost(schwab_sym: str) -> str:
    """'/MYMU26' -> 'MYMU2026'. Expands the 2-digit year to 4 digits."""
    s = schwab_sym.lstrip("/")
    root, mcode, yy = s[:-3], s[-3], s[-2:]
    return f"{root}{mcode}20{yy}"
 
 
# ── Client factories ────────────────────────────────────────────────────────
def build_client(asyncio_mode: bool = False):
    """Authenticated schwab-py client. Handles token fetch + auto-refresh.
 
    asyncio_mode=True is REQUIRED when you'll use the streaming client.
    """
    if not API_KEY or not APP_SECRET:
        raise RuntimeError("Set SCHWAB_API_KEY and SCHWAB_APP_SECRET in the environment.")
    return auth.easy_client(
        api_key=API_KEY,
        app_secret=APP_SECRET,
        callback_url=CALLBACK_URL,
        token_path=TOKEN_PATH,
        asyncio=asyncio_mode,
    )
 

In [ ]:
def get_live_quote_ohlc(client, symbol: str = SCHWAB_SYMBOL) -> dict:
    """Single REST quote -> current-session OHLC snapshot.
 
    Note on 'close': Schwab's `closePrice` is the PRIOR session's close, so for a
    live "current bar" we use `lastPrice` as the close. `openPrice/highPrice/
    lowPrice` are the running values for the current session.
 
    Returns a dict: {datetime, open, high, low, close, last, bid, ask, symbol}.
    Use get_quotes (NOT get_quote) for futures — '/' breaks URL-path encoding.
    """
    resp = client.get_quotes([symbol])
    resp.raise_for_status()
    payload = resp.json().get(symbol, {})
    q = payload.get("quote", {})
    if not q:
        raise RuntimeError(f"No quote returned for {symbol}. Check entitlement/symbol.")
 
    # quoteTime / tradeTime are epoch milliseconds when present.
    ts_ms = q.get("tradeTime") or q.get("quoteTime")
    ts = (datetime.fromtimestamp(ts_ms / 1000, tz=EASTERN)
          if ts_ms else datetime.now(tz=EASTERN))
 
    return {
        "datetime": ts,
        "open":  q.get("openPrice"),
        "high":  q.get("highPrice"),
        "low":   q.get("lowPrice"),
        "close": q.get("lastPrice"),       # live close = last traded price
        "last":  q.get("lastPrice"),
        "bid":   q.get("bidPrice"),
        "ask":   q.get("askPrice"),
        "symbol": symbol,
    }

# TradersPost Alert Workflow (Take Signals --> Send TraderPost Webhook --> Broker Order Execution)

In [1]:
# config.py — MYM Signal Alert System
# ---------------------------------------------------------------
# Secrets and environment-specific paths are loaded from the
# environment — NOT hard-coded here. Same file runs unchanged on
# your Mac (.env) and on the cloud host (Variables/Secrets UI).
# Never commit real secrets to this file.
# ---------------------------------------------------------------

import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # cloud host sets env vars directly

def _require(name):
    """Return a required env var, or fail loudly with a clear message."""
    val = os.getenv(name)
    if not val:
        raise RuntimeError(
            f"Missing required environment variable: {name}. "
            f"Set it in your local .env or the cloud host's environment. "
            f"See .env.example for the full list."
        )
    return val


# ── PATHS (environment-overridable) ───────────────────────────
BASE      = Path(os.getenv("MYM_BASE", os.path.expanduser("~/prod/em_algo")))
DATA_DIR  = BASE / "data"
DATA_1M   = str(DATA_DIR / "MYM_1m_tv_bento.csv")   # backtest base bars
LOG_FILE  = str(BASE / "logs" / "signal_log.txt")

# ── STRATEGY PARAMETERS (v6 — must match strategy.py backtest) ─
TICKER            = "MYM"
ACCOUNT_SIZE      = 50000
RISK_PCT          = 0.005      # 0.5% per trade (v7 backtest value)
TICK_VALUE        = 0.50       # MYM = $0.50/point (CME-verified; YM is $5)
MIN_SCORE         = 7
T2_CAP            = 2.5          # R:R cap on T2
MAX_BODY_PCT      = 30         # base candle: body <= 40% of range
MIN_BODY_PCT      = 35         # leg candle:  body >= 50% of range
MIN_RANGE_RATIO   = 0.631      # leg range >= base_height * this
RANGE_MULTIPLIER  = 0.36       # base range <= rolling-median * this
MIN_ZONE_HEIGHT   = 30         # pts
MAX_BASE_CANDLES  = 6
STOP_BUFFER       = 8          # static fallback; live run computes dynamically

# Lookbacks — note the UNITS, they differ:
ZONE_LOOKBACK_DAYS  = 60       # set_curve: last 60 x 15m bars
TREND_WINDOW        = 120       # classify_trend: rolling window of (last 10hrs) 5m bars
ZONE_WINDOW_DAYS    = 120       # detect_zones: last 120 candles days (last 10hrs) of 5m

# ── SCHEDULER ─────────────────────────────────────────────────
RUN_HOURS = [8, 9, 10, 11, 12]
TIMEZONE  = "US/Eastern"

# ── SECRETS (loaded from environment — never hard-coded) ──────
EMAIL_SENDER   = os.getenv("EMAIL_SENDER", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")   # Gmail APP password
EMAIL_RECEIVER = os.getenv("EMAIL_RECEIVER", EMAIL_SENDER)

# Discord: channel settings -> Integrations -> Webhooks -> copy URL
DISCORD_WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL")

# TradersPost: strategy webhook URL (used by alerts.traderspost_alert)
TRADERSPOST_WEBHOOK_URL = os.getenv("TRADERSPOST_WEBHOOK_URL", "")

# Live data providers (optional until activated)
ALPACA_KEY    = os.getenv("ALPACA_KEY", "")
ALPACA_SECRET = os.getenv("ALPACA_SECRET", "")

# Secrets required ONLY to send alerts. Enforced at send time, not import.
ALERT_REQUIRED = ["EMAIL_SENDER", "EMAIL_PASSWORD", "DISCORD_WEBHOOK_URL",
                  "TRADERSPOST_WEBHOOK_URL"]

def require_alert_secrets():
    """Raise if any alert secret is missing. Call before sending alerts."""
    missing = [k for k in ALERT_REQUIRED if not globals().get(k)]
    if missing:
        raise RuntimeError(
            f"Cannot send alert — missing secrets: {', '.join(missing)}. "
            f"Set them in your local .env or the cloud host's environment. "
            f"(Backtests/notebooks do not need these.)"
        )


In [2]:
# verify_env.py — confirm all required secrets are present.
# Prints only PRESENCE, never the values. Run after setting up your
# .env (local) or the host environment (cloud).
#
#   python verify_env.py

import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

REQUIRED = ["EMAIL_SENDER", "EMAIL_PASSWORD", "DISCORD_WEBHOOK_URL","TRADERSPOST_WEBHOOK_URL"]
OPTIONAL = ["EMAIL_RECEIVER", "MYM_BASE", "ALPACA_KEY", "ALPACA_SECRET",
            "MASSIVE_API_KEY", "DATABENTO_API_KEY"]

print("Required:")
for k in REQUIRED:
    print(f"  {'OK ' if os.getenv(k) else 'MISSING'}  {k}")

print("Optional:")
for k in OPTIONAL:
    print(f"  {'set' if os.getenv(k) else '- - '}  {k}")

missing = [k for k in REQUIRED if not os.getenv(k)]
if missing:
    raise SystemExit(f"\nFAIL — missing required: {', '.join(missing)}")

print("\nAll required secrets present. Safe to run signal_check.py.")


Required:
  OK   EMAIL_SENDER
  OK   EMAIL_PASSWORD
  OK   DISCORD_WEBHOOK_URL
  OK   TRADERSPOST_WEBHOOK_URL
Optional:
  set  EMAIL_RECEIVER
  set  MYM_BASE
  set  ALPACA_KEY
  set  ALPACA_SECRET
  set  MASSIVE_API_KEY
  - -   DATABENTO_API_KEY

All required secrets present. Safe to run signal_check.py.


In [3]:
# alerts.py - Valid Zone & Entry Signal Workflow

from __future__ import annotations

import os
import json
import logging
import smtplib
from dataclasses import dataclass
from typing import Any, Mapping
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import requests
from dotenv import load_dotenv
load_dotenv()
import sys
sys.path.insert(0, "/Users/efiles/prod/em_algo")
import config

logger = logging.getLogger(__name__)
# ─────────────────────────────────────────────
# alerts.py - MYM Signal Alert System
# Email via Gmail SMTP + Push notifications via Discord Webhook + TradersPost Send
# ─────────────────────────────────────────────

# DECLARE TEST MODE
# test_mode: when True, adds "test": true so TradersPost validates only.
test_mode = False

In [4]:
# ── TRADERSPOST PAPER TRADING ──────────────────────────────────
# """TradersPost webhook client for sending OTA trade signals.
test_mode = False
DEFAULT_TICKER = "MYM"
DEFAULT_TIMEOUT = 10  # seconds

# Signal types that mean "go long". Demand zones (drop-base-rally) buy;
# everything else sells. Extend this set as new long patterns are added.
LONG_SIGNAL_TYPES = frozenset({"DBR"})

# Internal signal keys that have no reserved TradersPost field and should be
# forwarded as opaque metadata under "extras".
EXTRA_KEYS = ("formed_at", "t2_points", "score", "stop_points", "total_risk", "potential_t1", "potential_t2")

# Internal signal keys that MUST be present to build a valid order.
REQUIRED_KEYS = ("type", "entry", "stop_points", "t2_points", "contracts")

# --------------------------------------------------------------------------- #
# Result type (so callers branch on structured data, never on log strings)
@dataclass(frozen=True)
class AlertResult:
    # """Outcome of a webhook send.

    # Attributes:
    #     ok:           True only on HTTP 200.
    #     status_code:  HTTP status, or None if the request never completed.
    #     signal_id:    TradersPost signal id from a 200 body (for reconciliation).
    #     message_code: Structured error code from a 4xx body (for branching).
    #     detail:       Human-readable message for logs/alerts.
    # """

    ok: bool
    status_code: int | None = None
    signal_id: str | None = None
    message_code: str | None = None
    detail: str = ""

# --------------------------------------------------------------------------- #
# Validation + payload construction (pure, no I/O — unit-testable in isolation)
# --------------------------------------------------------------------------- #

class SignalError(ValueError):
    """Raised when an internal signal cannot form a valid TradersPost order."""


def validate_signal(signal: Mapping[str, Any]) -> None:
    # """Fail fast, with context, before any network call.

    # Raises:
    #     SignalError: if a required key is missing.
    # """
    missing = [key for key in REQUIRED_KEYS if key not in signal]
    if missing:
        raise SignalError(f"signal missing required keys: {missing}")   

In [5]:
def build_payload(
    signal: Mapping[str, Any],
    *,
    ticker: str = DEFAULT_TICKER,
    test_mode: bool = test_mode,
) -> dict[str, Any]:
    # """Architecture B entry: full position, stop at `stop`, take-profit at the
    # runner target (t2 if a split, else t1). The T1 partial exit and breakeven
    # are fired separately at the T1-fill moment — see partial_exit_t1 / breakeven."""
    validate_signal(signal)

    entry  = signal["entry"]
    action = "buy" if signal["type"] in LONG_SIGNAL_TYPES else "sell"
    # Single-target signals (rr_raw <= 1.1) have t2 = None → target is t1.
    runner_target = signal["t2_points"] if signal.get("t2_points") is not None else signal["t1_points"]

    payload: dict[str, Any] = {
        "test": test_mode,
        "ticker": ticker,
        "action": action,
        "orderType": "limit",
        "price": entry,
        # no-quote brokers (Tradovate/Lucid) require a price on limit/bracket orders
        "signalPrice": entry,
        "quantity": signal["contracts"],
        "stopLoss":   {"type": "stop", "amount": signal["stop_points"]},   
        "takeProfit": {"amount": runner_target},                  
        "extras": {key: signal[key] for key in EXTRA_KEYS if key in signal},
    }
    
    if test_mode:
        payload["test"] = True

    return payload


def _parse_response(resp: requests.Response) -> AlertResult:
    """Map a raw HTTP response onto a structured AlertResult."""
    try:
        body = resp.json()
    except ValueError:
        body = {}

    if resp.status_code == 200:
        return AlertResult(
            ok=True,
            status_code=200,
            signal_id=body.get("id"),
            detail="accepted",
        )

    # 400/404/500: the spec provides messageCode + message on validation errors.
    return AlertResult(
        ok=False,
        status_code=resp.status_code,
        message_code=body.get("messageCode"),
        detail=body.get("message", resp.text),
    )

In [6]:
def traderspost_alert(
    signal: Mapping[str, Any],
    *,
    ticker: str = DEFAULT_TICKER,
    test_mode: bool = test_mode,
    timeout: int = DEFAULT_TIMEOUT,
    webhook_url: str | None = None,
) -> AlertResult:
    # """Send one OTA signal to the TradersPost webhook.

    # Args:
    #     signal:      internal signal mapping.
    #     ticker:      instrument symbol.
    #     test_mode:   if True, TradersPost validates but places no order.
    #     timeout:     seconds before the request is abandoned.
    #     webhook_url: override; defaults to env var TRADERSPOST_WEBHOOK_URL.

    # Returns:
    #     AlertResult — always returned, never raises on network/HTTP failure.
    #     (A SignalError from build_payload is the one intentional exception:
    #     a malformed signal is a programming error, not a runtime condition.)
    # """
    url = webhook_url or os.getenv("TRADERSPOST_WEBHOOK_URL")
    if not url:
        logger.error("TRADERSPOST_WEBHOOK_URL is not set")
        return AlertResult(ok=False, detail="missing webhook url")

    payload = build_payload(signal, ticker=ticker, test_mode=test_mode)
    logger.info("POST payload: %s", json.dumps(payload))

    try:
        resp = requests.post(url, json=payload, timeout=timeout)
    except requests.exceptions.Timeout:
        logger.error("Request timed out after %ss", timeout)
        return AlertResult(ok=False, detail=f"timeout after {timeout}s")
    except requests.exceptions.RequestException as exc:
        logger.error("Request failed: %s", exc)
        return AlertResult(ok=False, detail=str(exc))

    result = _parse_response(resp)
    if result.ok:
        logger.info("Accepted (200) signal_id=%s", result.signal_id)
    else:
        logger.warning(
            "Rejected (%s) code=%s: %s",
            result.status_code,
            result.message_code,
            result.detail,
        )
    return result


In [7]:
def partial_exit_t1(signal, *, ticker=DEFAULT_TICKER, test_mode=test_mode, timeout=DEFAULT_TIMEOUT):
    """Bank the T1 tranche: partial exit of contracts_t1 on the open position.
    To rest it as a limit at t1 instead of firing at the T1 moment, add
    orderType='limit' + limitPrice=signal['t1'] — VERIFY that combo works first."""
    config.require_alert_secrets()
    url = os.getenv("TRADERSPOST_WEBHOOK_URL")
    if not url:
        return AlertResult(ok=False, detail="missing webhook url")
    payload = {"test": test_mode, "ticker": ticker,
               "action": "exit", "quantity": signal["contracts_t1"]}
    logger.info("T1 PARTIAL EXIT %s", json.dumps(payload))
    try:
        resp = requests.post(url, json=payload, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        return AlertResult(ok=False, detail=str(exc))
    return _parse_response(resp)


In [9]:
def breakeven(ticker=DEFAULT_TICKER, *, test_mode=test_mode, timeout=DEFAULT_TIMEOUT):
    """Move the runner's stop to entry after T1 fills (needs an open position
    and the bracket's stop order)."""
    config.require_alert_secrets()
    url = os.getenv("TRADERSPOST_WEBHOOK_URL")
    if not url:
        return AlertResult(ok=False, detail="missing webhook url")
    payload = {
        "test": test_mode,
        "ticker": ticker,
        "action": "breakeven",
        "orderType": "stop",
        "cancel": True,
        "cancelOrderType": "stop",   # cancel the OLD stop only; leave the TP resting
    }    
    logger.info("BREAKEVEN %s", json.dumps(payload))
    try:
        resp = requests.post(url, json=payload, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        return AlertResult(ok=False, detail=str(exc))
    return _parse_response(resp)

In [10]:
def flatten(ticker=config.TICKER, *, test_mode=test_mode, timeout=DEFAULT_TIMEOUT):
    """Close any open position for `ticker`. Documented exit: action=exit, no
    quantity (=> full position), ignoreTradingWindows so a window can't block it."""
    config.require_alert_secrets()
    url = os.getenv("TRADERSPOST_WEBHOOK_URL")
    if not url:
        logger.error("TRADERSPOST_WEBHOOK_URL is not set")
        return AlertResult(ok=False, detail="missing webhook url")
    payload = {"ticker": ticker, "action": "exit", "cancel": True, "ignoreTradingWindows": True}
    if test_mode:
        payload["test"] = test_mode
    logger.info("FLATTEN %s", json.dumps(payload))
    try:
        resp = requests.post(url, json=payload, timeout=timeout)
    except requests.exceptions.RequestException as exc:
        logger.error("Flatten failed: %s", exc)
        return AlertResult(ok=False, detail=str(exc))
    return _parse_response(resp)


        # alert_data = {
        #     "type"        : zone["type"],
        #     "entry"       : sets["entry"],
        #     "stop"        : sets["stop"],
        #     "t1"          : sets["t1"],
        #     "t2"          : sets["t2"],
        #     "stop_points" : sets["stop_points"],
        #     "rr_t1"       : sets["rr_t1"],
        #     "rr_t2"       : sets["rr_t2"],
        #     "t1_points"   : sets["t1_points"],
        #     "t2_points"   : sets["t2_points"],
        #     "contracts"   : sets["contracts"],
        #     "score"       : score,
        #     "untouched?"  : zone["touched"],
        #     "formed_at"   : zone["formed_at"],
        #     "total_risk"  : sets["total_risk_$"],
        #     "potential_t1": sets["potential_t1_$"],
        #     "potential_t2": sets["potential_t2_$"],
        # }

In [12]:
sig = {
    "type": "DBR", "entry": 47800,
    "stop_points": 38,          # → stopLoss.amount
    "t1_points": 171,           # single-target offset
    "t2_points": 380,           # → takeProfit.amount (runner)
    "contracts": 2, "contracts_t1": 1, "contracts_t2": 1,
}


In [13]:
# 1. Entry — full 2 contracts, stop 47762, runner TP at t2 48180
traderspost_alert(sig, test_mode=False)
#    → check positions: 2 long? one stop at 47762? one TP at 48180?

AlertResult(ok=True, status_code=200, signal_id='7e65e57a-63fa-4db3-ab66-26dfa25bffb3', message_code=None, detail='accepted')

In [14]:
# 2. Bank T1 — partial exit of 1 contract
partial_exit_t1(sig, test_mode=False)
#    → check: position now 1? does the resting stop/TP auto-reduce to qty 1,
#      or still show qty 2 (dangling)? ← KEY OBSERVATION

AlertResult(ok=True, status_code=200, signal_id='eabc03f4-d1ea-4149-a94d-3dd0c25751ed', message_code=None, detail='accepted')

In [15]:
# 3. Breakeven — move runner stop to entry
breakeven(test_mode=False)
#    → check: stop moves 47762 → 47800? position STILL OPEN (1 contract)?
#      if the position closed, breakeven misbehaved ← KEY OBSERVATION

AlertResult(ok=True, status_code=200, signal_id='8cb5d4e5-571a-41fb-83cd-b116a3d60eeb', message_code=None, detail='accepted')

In [ ]:
# 4. Cleanup so the test leaves nothing resting
flatten(test_mode=False)

In [ ]:
# ── COMBINED ALERT ────────────────────────────

def send_alert(zone, *, test_mode: bool = test_mode):
    # When ready for real signals to be sent for orders, change test_mode: to bool = True
    # Enforce that alert secrets exist ONLY when actually sending.
    # config.py reads secrets lazily (empty if unset) so backtests/notebooks
    # can import config freely; this is where the requirement is enforced.
    config.require_alert_secrets()
    direction = "BUY  🟢" if zone["type"] == "DBR" else "SELL 🔴"

    subject = f"MYM Signal — {direction} @ {zone['entry']}"

    email_body = f"""
MYM SIGNAL ALERT
─────────────────────────
Direction  : {direction}
Zone Type  : {zone['type']}
Entry      : {zone['entry']}
Stop       : {zone['stop']}
T1 Target  : {zone['t1']}
T2 Target  : {zone['t2']}
Stop Pts   : {zone['stop_points']}
T1 Points  : {zone['t1_points']}
T2 Points  : {zone['t2_points']}
R:R T1     : {zone['rr_t1']}:1
R:R T2     : {zone['rr_t2']}:1
Contracts  : {zone['contracts']}
Score      : {zone['score']}
Formed At  : {zone['formed_at']}
─────────────────────────
Max Risk   : ${zone['total_risk']}
T1 Profit  : ${zone['potential_t1']}
T2 Profit  : ${zone['potential_t2']}
"""
    # Concise push notification
    message = (
        f"{zone['type']} Zone\n"
        f"Entry     : {zone['entry']}\n"
        f"Stop      : {zone['stop']}\n"
        f"T1        : {zone['t1']}\n"
        f"T2        : {zone['t2']}\n"
        f"Formed At : {zone['formed_at']}\n"
        f"Score     : {zone['score']} | R:R {zone['rr_t1']}:1\n"
        f"Risk      : ${zone['total_risk']}\n"
        f"Contracts : {zone['contracts']}\n"  
        f"Stop Pts  : {zone['stop_points']}\n"
        f"T1 Points : {zone['t1_points']}\n"
        f"T2 Points : {zone['t2_points']}\n"
    )
        
    #send_email(subject, email_body)
    #discord_alert(message, alert_type="signal")
    return traderspost_alert(zone, test_mode=test_mode)
     
# ── TEST FUNCTION ─────────────────────────────

def test_alerts():
    print("Sending test alert...")
    test_zone = {
        "type"        : "DBR",
        "entry"       : 47800,
        "stop"        : 47762,
        "t1"          : 47971,
        "t2"          : 48180,
        "stop_points" : 38,
        "t1_points"   : 171,
        "t2_points"   : 380,
        "rr_t1"       : 4.5,
        "rr_t2"       : 10.0,
        "contracts"   : 2,
        "contracts_t1": 1,
        "contracts_t2": 1,
        "score"       : 9.0,
        "untouched?"  : 0,
        "formed_at"   : "2026-03-24 09:30:00",
        "total_risk"  : 38.00,
        "potential_t1": 171.00,
        "potential_t2": 380.00
    }

    send_alert(test_zone)

if __name__ == "__main__":
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )
    test_alerts()
